<a href="https://colab.research.google.com/github/gabrielhierro/LinguagensDeProgramacao/blob/main/ExercicioPraticoPandas/Exercicio_Pratico_Pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd

# Base de vendas
dados_vendas = {
    'cliente_id': [101, 102, 103, 101, 104, 102, 105, 103],
    'valor': [3500.75, 189.50, np.nan, 1200.00, 450.00, np.nan, 89.90, 780.50],
    'categoria': [
        'Eletronicos',
        'Livros',
        'Roupas',
        'Eletronicos',
        'Automotivo',
        'Livros',
        'Roupas',
        'Roupas',
    ],
    'data_hora': [
        '2024-01-15 10:23:00',
        '2024-01-18 14:05:00',
        '2024-02-05 09:12:00',
        '2024-02-20 16:40:00',
        '2024-03-02 11:00:00',
        '2024-03-15 18:30:00',
        '2024-04-10 08:20:00',
        '2024-04-22 13:45:00',
    ],
    'status': [
        'Concluído',
        'Concluído',
        'Pendente',
        'Concluído',
        'Cancelado',
        'Concluído',
        'Concluído',
        'Concluído',
    ],
    'email': [
        'maria@gmail.com',
        'joao@outlook.com',
        'ana@yahoo.com',
        'maria@gmail.com',
        'carlos@gmail.com',
        'joao@outlook.com',
        'lucas@empresa.com.br',
        'ana@yahoo.com',
    ],
}

# Base de clientes
dados_clientes = {
    'cliente_id': [101, 102, 103, 104, 105],
    'nome': [
        'Maria Silva',
        'Joao Souza',
        'Ana Oliveira',
        'Carlos Lima',
        'Lucas Mendes',
    ],
    'cidade': [
        'Sao Paulo',
        'Rio de Janeiro',
        'Belo Horizonte',
        'Curitiba',
        'Salvador',
    ],
}

df_vendas = pd.DataFrame(dados_vendas)
df_clientes = pd.DataFrame(dados_clientes)

## Parte 1 - Diagnóstico e Limpeza
1. Inspecione as dimensões e o resumo dos tipos de dados do DataFrame df_vendas usando .shape e .info().


In [6]:
linhas, colunas = df_vendas.shape
print(f'Dimensões: {linhas} linhas x {colunas} colunas')

Dimensões: 8 linhas x 6 colunas


In [5]:
df_vendas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   cliente_id  8 non-null      int64  
 1   valor       6 non-null      float64
 2   categoria   8 non-null      object 
 3   data_hora   8 non-null      object 
 4   status      8 non-null      object 
 5   email       8 non-null      object 
dtypes: float64(1), int64(1), object(4)
memory usage: 516.0+ bytes


2. Identifique os valores nulos e preencha os registros ausentes da coluna valor utilizando a mediana da respectiva categoria ou a mediana global com .fillna().


In [7]:
df_vendas['valor'] = df_vendas['valor'].fillna(df_vendas['valor'].median())
df_vendas['valor'].isna().sum()  # 0 = sem valores nulos restantes

np.int64(0)

3. Filtre e exiba apenas as transações com status 'Concluído' e valor superior a R$ 500,00 usando .loc[] ou .query().


In [10]:
df_vendas.loc[(df_vendas['valor'] > 500) & (df_vendas['status'] == 'Concluído'), ['cliente_id', 'valor', 'status']]

,cliente_id,valor,status
0,101,3500.75,Concluído
3,101,1200.00,Concluído
5,102,615.25,Concluído
7,103,780.50,Concluído


## Parte 2 - Cruzamento e Transformação
4. Realize uma junção relacional (left merge) entre df_vendas e df_clientes utilizando a chave cliente_id.
5. Crie uma nova coluna chamada media_categoria contendo o valor médio de vendas por categoria sem reduzir o número de linhas do DataFrame (utilize .groupby() com .transform()).
6. Identifique quantos clientes usam o provedor de e-mail @gmail.com através do acessor de texto .str.contains().

In [13]:
## Preparação
vendas = df_vendas.copy()
clientes = df_clientes.copy()

In [12]:
# Tarefa 4:
df_completo = pd.merge(vendas, clientes, on='cliente_id', how='left')
df_completo
df_completo.shape

(8, 8)

In [15]:
# Tarefa 5:
df_vendas['media_categoria'] = df_vendas.groupby('categoria')['valor'].transform('mean')
df_vendas[['categoria', 'valor', 'media_categoria']]

,categoria,valor,media_categoria
0,Eletronicos,3500.75,2350.375000
1,Livros,189.50,402.375000
2,Roupas,615.25,495.216667
3,Eletronicos,1200.00,2350.375000
4,Automotivo,450.00,450.000000
5,Livros,615.25,402.375000
6,Roupas,89.90,495.216667
7,Roupas,780.50,495.216667


In [17]:
# Tarefa 6:
gmail = df_vendas[df_vendas['email'].str.contains('@gmail.com')]
print('Emails do Gmail:', len(gmail))

Emails do Gmail: 3


## Parte 3 - Análise Temporal e Agregação
7. Converta a coluna data_hora para o tipo datetime nativo com pd.to_datetime() e extraia o mês por extenso ou o nome do dia da semana utilizando o acessor .dt.
8. Construa uma tabela dinâmica com pd.pivot_table() apresentando o total (sum) do valor de vendas agrupado por categoria nas linhas e por cidade nas colunas, preenchendo eventuais valores nulos com zero.

In [19]:
# Tarefa 7:
df_vendas['data_hora'] = pd.to_datetime(df_vendas['data_hora'], format='%Y-%m-%d %H:%M:%S')
df_vendas['data_hora']

df_vendas['ano'] = df_vendas['data_hora'].dt.year
df_vendas['dia_semana'] = df_vendas['data_hora'].dt.day_name()
df_vendas[['data_hora', 'ano', 'dia_semana']]

,data_hora,ano,dia_semana
0,2024-01-15 10:23:00,2024,Monday
1,2024-01-18 14:05:00,2024,Thursday
2,2024-02-05 09:12:00,2024,Monday
3,2024-02-20 16:40:00,2024,Tuesday
4,2024-03-02 11:00:00,2024,Saturday
5,2024-03-15 18:30:00,2024,Friday
6,2024-04-10 08:20:00,2024,Wednesday
7,2024-04-22 13:45:00,2024,Monday


In [20]:
# Tarefa 8:
pd.pivot_table(df_completo, values='valor', index='categoria', columns='cidade', aggfunc='sum', fill_value=0)

cidade,Belo Horizonte,Curitiba,Rio de Janeiro,Salvador,Sao Paulo
categoria,,,,,
Automotivo,0.00,450.0,0.00,0.0,0.00
Eletronicos,0.00,0.0,0.00,0.0,4700.75
Livros,0.00,0.0,804.75,0.0,0.00
Roupas,1395.75,0.0,0.00,89.9,0.00
